# 매출예측 DB 마이그레이션 — `Korea_revenue_forecast_DB_migration_v2` (1회 실행용)

`korea_revenue_forecast_result` 테이블에 누적된 세 가지 문제를 **한 번만** 정리하는 노트북입니다.
정리 후에는 이 노트북을 다시 열 일이 없습니다. 일상적인 매출 예측은
`Korea_revenue_forecast_v6.ipynb` 만 사용하세요.

### 이 노트북이 하는 일 / 하지 않는 일

| 하는 일 | 하지 않는 일 |
|---|---|
| 기존 테이블의 UNIQUE KEY 를 vintage 4열 키로 교정 (`ALTER TABLE`) | **새 DB · 새 테이블을 만들지 않음** — 기존 `investar.korea_revenue_forecast_result` 를 제자리에서 수리 |
| 레거시 행의 단위·티커 표기 수정 (`UPDATE`) | 예측값 재계산 · 행 삭제 없음 |

### 실행 순서 — 반드시 ① 키 → ② 단위 → ③ 티커

티커를 먼저 통일하면 구세대 행('005930')과 신세대 행('A005930')이 같은
`(date, ticker, indicator)` 키로 충돌할 수 있습니다. 키에 `created_at` 이 먼저
들어가면 세대가 다른 행은 `created_at` 이 달라 충돌하지 않습니다.

| 단계 | 함수 | 성격 |
|---|---|---|
| ⓪ 진단 | `audit_result_table()` | **읽기 전용** — collation 정합 · 키 구성 · 티커 형식 분포 · 세대별 값 자릿수 · 원천 실적 대비 단위 교차검증 |
| ① 키 | `migrate_vintage_key()` | 멱등 (예측 노트북 v6의 저장 함수도 매 실행 시 자동 수행) |
| ② 단위 | `migrate_unit(factor=...)` | `factor` 는 진단 결과를 보고 **직접 지정**. 기본 `dry_run=True` |
| ③ 티커 | `migrate_ticker()` | 충돌 사전 검사 후 표준 형식으로 통일. 기본 `dry_run=True` |
| ④ collation | `migrate_collation()` | 결과 테이블의 문자셋/콜레이션을 원천 FS 테이블과 통일. 멱등, ①~③ 과 순서 무관. 기본 `dry_run=True` |

> ⚠️ 아래 설정 셀의 `RESULT_TICKER_HAS_A_PREFIX` 는 예측 노트북(v6)의 같은 설정과
> **반드시 같은 값**이어야 합니다. 기본값은 두 노트북 모두 `False` ('005930' 형)입니다.

### v2 변경 사항 (collation 불일치 1267 대응)

`korea_revenue_forecast_result` 와 `korea_fs_data_from_DG` 의 collation 이 달라
(`utf8mb4_general_ci` vs `utf8mb4_unicode_ci`) ticker 조인에서
`OperationalError 1267: Illegal mix of collations` 가 발생했다.

1. 진단 `[0]` 신설 — 두 테이블 ticker 컬럼의 collation 을 먼저 표시
2. 진단 `[4]` 조인에 명시적 `COLLATE` 를 추가해 불일치 상태에서도 읽기 전용 진단이 동작
3. `④ migrate_collation()` 신설 — 결과 테이블을 원천 FS 테이블의 collation 으로 통일 (근본 해결)

티커는 ASCII 문자열이므로 general/unicode 어느 쪽으로 비교해도 결과는 동일하며,
통일의 목적은 valuation 파이프라인 전반의 조인에서 같은 에러가 재발하지 않게 하는 것이다.


## 1. 환경 설정

In [1]:
import sys
import os
import pandas as pd
import pymysql
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')


In [2]:
def setup_universal_paths():
    """
    어떤 PC에서도 작동하는 범용 경로 설정 v2 — DATA 폴더 자동 탐색.
    탐색 순서:
      ① 환경변수 DATA_DIR (직접 지정 시)
      ② 현재 폴더와 상위 폴더들의 직속 DATA
      ③ 현재 폴더 하위(깊이 3) 및 상위 3단계 폴더들의 하위(깊이 2)
    old_files/backup 등 백업 폴더 내의 DATA 는 무시한다.
    이 노트북은 대만 프로젝트(config.py)와 무관하며 DATA 폴더만 있으면 동작한다.
    """
    import os
    SKIP = {".git", ".idea", ".venv", "venv", "env", "__pycache__",
            "node_modules", "site-packages", ".ipynb_checkpoints",
            "AppData", "anaconda3", "Miniconda3",
            "old_files", "old", "_old", "backup", "backups", "archive", "bak"}
    current = Path.cwd()

    def register(data_folder):
        parent = data_folder.parent
        for p in (str(parent), str(data_folder)):
            if p not in sys.path:
                sys.path.insert(0, p)
        print("=" * 80)
        print("경로 설정 완료")
        print("=" * 80)
        print(f"프로젝트 루트: {parent}")
        print(f"DATA 폴더:    {data_folder}")
        print(f"현재 위치:     {current}")
        print("=" * 80 + "\n")
        return {"project_root": parent, "data_folder": data_folder, "current": current}

    # ① 환경변수
    env = os.environ.get("DATA_DIR")
    if env and Path(env).exists():
        return register(Path(env))

    # ② 상위 방향: 직속 DATA
    for parent in [current, *current.parents]:
        d = parent / "DATA"
        if d.exists():
            return register(d)

    # ③ 하위 방향: cwd 깊이 3, 상위 3단계는 깊이 2
    def scan(root, max_depth):
        root = Path(root); base = len(root.parts)
        for dp, dn, _ in os.walk(root):
            if len(Path(dp).parts) - base >= max_depth:
                dn[:] = []
            dn[:] = [x for x in dn if x not in SKIP and not x.startswith(".")]
            if "DATA" in dn:
                return Path(dp) / "DATA"
        return None
    hit = scan(current, 3)
    if hit is None:
        for parent in list(current.parents)[:3]:
            hit = scan(parent, 2)
            if hit:
                break
    if hit:
        return register(hit)

    raise FileNotFoundError(
        f"DATA 폴더를 찾을 수 없습니다.\n현재 위치: {current}\n"
        "아래처럼 직접 지정한 뒤 이 셀을 다시 실행하세요:\n"
        '  import os; os.environ["DATA_DIR"] = r"C:\\...\\DATA"'
    )


# 경로 설정 실행
paths = setup_universal_paths()

# DATA 공용 모듈에서 DB 호스트 판별 함수 로드
from stock_invest_function import get_db_host
print("get_db_host import 성공")


경로 설정 완료
프로젝트 루트: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
DATA 폴더:    C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
현재 위치:     C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\Korea_Market\analysis\한국기업_매출예측

get_db_host import 성공


In [3]:
# ==================== 대상 테이블 · 표준 설정 ====================
# 예측 노트북(Korea_revenue_forecast_v6)의 동일 설정과 반드시 일치시킬 것

RESULT_TABLE = "korea_revenue_forecast_result"   # 수리 대상 (기존 테이블, 새로 만들지 않음)

# 결과 테이블의 ticker 표준 형식: False='005930' / True='A005930'
RESULT_TICKER_HAS_A_PREFIX = False

# 단위 교차검증용 원천 (DataGuide 실적, 단위: 천원)
FS_TABLE = "korea_fs_data_from_DG"
REVENUE_ITEM_CODE = "M000904001"


In [4]:
def get_db_info() -> dict:
    """DB 연결 정보 반환"""
    return {
        "host": get_db_host(),
        "port": 3307,
        "user": "stox7412",
        "password": "Apt106503!~",
        "database": "investar",
    }


def get_connection(db_info: dict):
    """DB 연결 생성 (pymysql)"""
    return pymysql.connect(
        host=db_info["host"],
        port=int(db_info["port"]),
        user=db_info["user"],
        password=db_info["password"],
        db=db_info["database"],
        charset="utf8mb4",
    )

## 2. 진단 · 마이그레이션 함수

In [5]:
VINTAGE_KEY_COLS = "date,ticker,indicator,created_at"
LEGACY_KEY_COLS = "date,ticker,indicator"


def ensure_vintage_key(cursor, table_name: str) -> str:
    """
    실제 DB 테이블의 UNIQUE KEY 가 vintage 4열 키인지 검사하고, 아니면 교정한다 (멱등).

    v4까지의 문제: 코드의 CREATE TABLE IF NOT EXISTS 에는 4열 키가 있었지만,
    테이블이 이미 존재하면 이 문장은 아무것도 하지 않는다. 구버전이 만든
    (date, ticker, indicator) 3열 키가 살아 있으면 재실행 때마다 과거 예측이
    덮어써져 vintage 가 소멸한다. 매 저장 전에 여기서 보증한다.

    Returns: 'ok' | 'altered' | 'created'
    """
    cursor.execute(
        """
        SELECT INDEX_NAME, GROUP_CONCAT(COLUMN_NAME ORDER BY SEQ_IN_INDEX) AS cols
        FROM information_schema.STATISTICS
        WHERE TABLE_SCHEMA = DATABASE()
          AND TABLE_NAME = %s
          AND NON_UNIQUE = 0
          AND INDEX_NAME <> 'PRIMARY'
        GROUP BY INDEX_NAME
        """,
        (table_name,),
    )
    uniques = {name: cols for name, cols in cursor.fetchall()}

    if VINTAGE_KEY_COLS in uniques.values():
        return "ok"

    legacy = [n for n, c in uniques.items() if c == LEGACY_KEY_COLS]
    if legacy:
        # 3열 키가 unique 였다면 4열 조합은 자동으로 unique 이므로 ALTER 는 안전하다.
        cursor.execute(
            f"ALTER TABLE {table_name} "
            f"DROP INDEX `{legacy[0]}`, "
            f"ADD UNIQUE KEY unique_date_ticker_indicator_created "
            f"(date, ticker, indicator, created_at)"
        )
        print(f"  [v5] vintage 키 교정: `{legacy[0]}`({LEGACY_KEY_COLS}) -> (date,ticker,indicator,created_at)")
        return "altered"

    # unique 키가 아예 없던 경우 — 중복이 있으면 여기서 실패하며, 그때는 §6 감사로 원인 확인
    cursor.execute(
        f"ALTER TABLE {table_name} "
        f"ADD UNIQUE KEY unique_date_ticker_indicator_created "
        f"(date, ticker, indicator, created_at)"
    )
    print("  [v5] vintage 키 신규 추가: (date,ticker,indicator,created_at)")
    return "created"


def _fetch_df(conn, sql, params=None):
    return pd.read_sql(sql, conn, params=params)


def audit_result_table(db_info: dict, table_name: str = RESULT_TABLE, n_sample: int = 30) -> None:
    """
    결과 테이블 현황 진단 (읽기 전용).
      0) ticker 컬럼 collation 정합 (원천 FS 테이블 대비) — 불일치 시 조인 1267 에러의 원인
      1) UNIQUE KEY 구성
      2) ticker 형식 분포 ('A' 접두사 유무)
      3) created_at 세대별 요약 (행수, 값 자릿수 중앙값)
      4) 단위 교차검증: 앙상블 예측 vs 원천 실적(천원)의 log10 자릿수 차이
         -> 차이 ~ 0 이면 원천과 같은 '천원', ~ +3 이면 1000배(원 단위) 의심
    """
    conn = get_connection(db_info)
    try:
        print("=" * 78)
        print(f"AUDIT: {table_name}")
        print("=" * 78)

        # 0) collation 정합 — 불일치 상태면 ticker 조인마다 1267 이 터진다
        coll = _fetch_df(conn, """
            SELECT TABLE_NAME, COLUMN_NAME, CHARACTER_SET_NAME, COLLATION_NAME
            FROM information_schema.COLUMNS
            WHERE TABLE_SCHEMA = DATABASE()
              AND TABLE_NAME IN (%s, %s)
              AND COLUMN_NAME = 'ticker'
        """, [table_name, FS_TABLE])
        print("\n[0] ticker 컬럼 collation (원천 FS 테이블과 일치해야 함)")
        print(coll.to_string(index=False))
        if coll["COLLATION_NAME"].nunique() > 1:
            print("    -> 불일치: 아래 [4] 는 명시적 COLLATE 로 우회 중. ④ migrate_collation 으로 통일 권장")

        # 1) unique key
        keys = _fetch_df(conn, """
            SELECT INDEX_NAME,
                   GROUP_CONCAT(COLUMN_NAME ORDER BY SEQ_IN_INDEX) AS cols
            FROM information_schema.STATISTICS
            WHERE TABLE_SCHEMA = DATABASE() AND TABLE_NAME = %s
              AND NON_UNIQUE = 0 AND INDEX_NAME <> 'PRIMARY'
            GROUP BY INDEX_NAME
        """, [table_name])
        print("\n[1] UNIQUE KEY")
        if keys.empty:
            print("    (없음) — vintage 보존 불가 상태")
        else:
            for _, r in keys.iterrows():
                mark = "OK (vintage)" if r["cols"] == VINTAGE_KEY_COLS else "구버전 — 실행 시 덮어쓰기 발생"
                print(f"    {r['INDEX_NAME']}: ({r['cols']})  <- {mark}")

        # 2) ticker 형식
        fmt = _fetch_df(conn, f"""
            SELECT CASE WHEN ticker LIKE 'A%%' THEN 'A접두사(예: A005930)'
                        ELSE '숫자만(예: 005930)' END AS fmt,
                   COUNT(*) AS n_rows, COUNT(DISTINCT ticker) AS n_tickers
            FROM {table_name} GROUP BY 1
        """)
        print("\n[2] ticker 형식 분포")
        print(fmt.to_string(index=False))
        if len(fmt) > 1:
            print("    -> 형식 혼재: 이 상태에서는 단일 형식 조인이 조용히 일부를 놓친다")

        # 3) 세대별 요약
        gen = _fetch_df(conn, f"""
            SELECT created_at,
                   CASE WHEN ticker LIKE 'A%%' THEN 'A접두사' ELSE '숫자만' END AS fmt,
                   COUNT(*) AS n_rows,
                   ROUND(AVG(LOG10(NULLIF(ABS(value), 0))), 2) AS avg_log10_value
            FROM {table_name}
            GROUP BY created_at, fmt ORDER BY created_at, fmt
        """)
        print("\n[3] created_at 세대별 요약 (avg_log10_value 가 세대 간 ~3 차이나면 1000배 의심)")
        print(gen.to_string(index=False))

        # 4) 원천 실적 대비 자릿수 교차검증 (표본)
        xchk = _fetch_df(conn, f"""
            SELECT CASE WHEN f.ticker LIKE 'A%%' THEN 'A접두사' ELSE '숫자만' END AS fmt,
                   COUNT(DISTINCT f.ticker) AS n_tickers,
                   ROUND(AVG(f.lg) - AVG(a.lg), 2) AS log10_gap_forecast_minus_actual
            FROM (
                SELECT ticker, AVG(LOG10(NULLIF(ABS(value),0))) AS lg
                FROM {table_name} WHERE indicator = 'Ensemble'
                GROUP BY ticker ORDER BY RAND() LIMIT %s
            ) f
            JOIN (
                SELECT ticker, AVG(LOG10(NULLIF(ABS(value),0))) AS lg
                FROM {FS_TABLE} WHERE item_code = %s
                GROUP BY ticker
            ) a
              ON a.ticker = CONCAT('A', LPAD(TRIM(LEADING 'A' FROM f.ticker), 6, '0'))
                            COLLATE utf8mb4_unicode_ci
            GROUP BY fmt
        """, [n_sample * 4, REVENUE_ITEM_CODE])
        print(f"\n[4] 원천 실적(천원) 대비 log10 차이 (표본, Ensemble 기준)")
        if xchk.empty:
            print("    (표본 조인 실패 — FS_TABLE 접근을 확인)")
        else:
            print(xchk.to_string(index=False))
            print("    해석: ~0.0 = 원천과 동일 단위(천원) · ~+3.0 = 1000배 큼(원 단위로 저장된 세대)")
        print("=" * 78)
    finally:
        conn.close()


def migrate_vintage_key(db_info: dict, table_name: str = RESULT_TABLE) -> None:
    """① vintage 키 보증 (멱등). save_forecasts_batch() 도 매 실행 시 자동 수행."""
    conn = get_connection(db_info)
    try:
        cur = conn.cursor()
        status = ensure_vintage_key(cur, table_name)
        conn.commit()
        print(f"vintage 키 상태: {status}")
    finally:
        conn.close()


def migrate_unit(
        db_info: dict,
        factor: float,
        where_legacy: str = "ticker NOT LIKE 'A%'",
        table_name: str = RESULT_TABLE,
        dry_run: bool = True,
) -> None:
    """
    ② 레거시 행 단위 통일: value <- value * factor.

    factor 는 감사(§6-⓪) 결과를 보고 직접 지정한다.
      예) 레거시가 '원', 표준이 '천원'  -> factor = 0.001
          레거시가 '천원', 표준이 '원'  -> factor = 1000
    where_legacy 로 대상 세대를 지정한다 (기본: 'A' 접두사 없는 구세대 행).
      created_at 으로 지정하려면 예: "created_at < '2026-08-26'"
    ※ 반드시 티커 통일(③) '이전'에 실행 — 티커 형식이 세대 구분자다.
    """
    conn = get_connection(db_info)
    try:
        # 파라미터와 함께 실행하는 쿼리에서는 LIKE 의 '%' 를 '%%' 로 이스케이프해야 한다 (pymysql)
        esc_where = where_legacy.replace("%", "%%")
        n = _fetch_df(conn, f"SELECT COUNT(*) AS n FROM {table_name} WHERE {where_legacy}")["n"].iloc[0]
        sample = _fetch_df(conn, f"""
            SELECT id, date, ticker, indicator, value, value * %s AS value_after
            FROM {table_name} WHERE {esc_where} LIMIT 5
        """, [factor])
        print(f"대상: {n:,}행  (WHERE {where_legacy})  factor={factor}")
        print(sample.to_string(index=False))
        if dry_run:
            print("\n[dry run] 실제 변경 없음. 확인 후 dry_run=False 로 재실행.")
            return
        cur = conn.cursor()
        cur.execute(f"UPDATE {table_name} SET value = value * %s WHERE {esc_where}", [factor])
        conn.commit()
        print(f"\nUPDATE 완료: {cur.rowcount:,}행")
    finally:
        conn.close()


def migrate_ticker(
        db_info: dict,
        table_name: str = RESULT_TABLE,
        dry_run: bool = True,
) -> None:
    """
    ③ ticker 표기 통일 (RESULT_TICKER_HAS_A_PREFIX 기준).
    실행 전 충돌 검사: 정규화 후 같은 (date, ticker, indicator, created_at)이 되는
    행 쌍이 있으면 중단하고 목록을 출력한다.
    ※ 반드시 ①(키) 이후에 실행.
    """
    conn = get_connection(db_info)
    try:
        collisions = _fetch_df(conn, f"""
            SELECT a.id AS id_a, b.id AS id_b, a.date, a.indicator, a.created_at,
                   a.ticker AS ticker_a, b.ticker AS ticker_b
            FROM {table_name} a
            JOIN {table_name} b
              ON a.id < b.id
             AND a.date = b.date
             AND a.indicator = b.indicator
             AND a.created_at = b.created_at
             AND TRIM(LEADING 'A' FROM a.ticker) = TRIM(LEADING 'A' FROM b.ticker)
             AND a.ticker <> b.ticker
            LIMIT 20
        """)
        if not collisions.empty:
            print("충돌 발견 — 정규화하면 UNIQUE KEY 가 겹치는 행 쌍이 있습니다. 처리 후 재시도:")
            print(collisions.to_string(index=False))
            return

        if RESULT_TICKER_HAS_A_PREFIX:
            where, expr = "ticker NOT LIKE 'A%'", "CONCAT('A', LPAD(ticker, 6, '0'))"
        else:
            where, expr = "ticker LIKE 'A%'", "LPAD(TRIM(LEADING 'A' FROM ticker), 6, '0')"

        n = _fetch_df(conn, f"SELECT COUNT(*) AS n FROM {table_name} WHERE {where}")["n"].iloc[0]
        target = "'A005930' 형" if RESULT_TICKER_HAS_A_PREFIX else "'005930' 형"
        print(f"충돌 없음. 변환 대상 {n:,}행 -> 표준 {target}")
        if dry_run:
            print("[dry run] 실제 변경 없음. 확인 후 dry_run=False 로 재실행.")
            return
        cur = conn.cursor()
        cur.execute(f"UPDATE {table_name} SET ticker = {expr} WHERE {where}")
        conn.commit()
        print(f"UPDATE 완료: {cur.rowcount:,}행")
    finally:
        conn.close()


def migrate_collation(
        db_info: dict,
        table_name: str = RESULT_TABLE,
        anchor_table: str = FS_TABLE,
        dry_run: bool = True,
) -> None:
    """
    ④ collation 통일 (멱등, ①~③ 과 순서 무관).

    결과 테이블의 문자셋/콜레이션을 앵커(원천 FS) 테이블의 ticker 컬럼 기준으로
    일치시킨다. 불일치 상태에서는 ticker 조인마다 1267 (Illegal mix of collations)
    이 발생하며, 진단 [4] 뿐 아니라 valuation 파이프라인의 모든 조인이 대상이다.

    CONVERT TO 는 테이블의 모든 문자열 컬럼을 일괄 변환하고 테이블을 재작성한다
    (행 수에 비례한 시간 소요, 실행 중 테이블 잠금). 값 자체는 ASCII 라 변형 없음.
    """
    conn = get_connection(db_info)
    try:
        cols = _fetch_df(conn, """
            SELECT TABLE_NAME, COLUMN_NAME, CHARACTER_SET_NAME, COLLATION_NAME
            FROM information_schema.COLUMNS
            WHERE TABLE_SCHEMA = DATABASE()
              AND TABLE_NAME IN (%s, %s)
              AND COLLATION_NAME IS NOT NULL
        """, [table_name, anchor_table])
        anchor = cols[(cols["TABLE_NAME"] == anchor_table) & (cols["COLUMN_NAME"] == "ticker")]
        if anchor.empty:
            print(f"앵커 테이블 {anchor_table} 의 ticker 컬럼을 찾지 못했습니다.")
            return
        charset = anchor["CHARACTER_SET_NAME"].iloc[0]
        target = anchor["COLLATION_NAME"].iloc[0]

        mine = cols[cols["TABLE_NAME"] == table_name]
        print(f"앵커: {anchor_table}.ticker = {charset} / {target}")
        print(f"\n{table_name} 문자열 컬럼 현황:")
        print(mine[["COLUMN_NAME", "CHARACTER_SET_NAME", "COLLATION_NAME"]].to_string(index=False))

        mismatched = mine[mine["COLLATION_NAME"] != target]
        if mismatched.empty:
            print(f"\n이미 일치: 모든 문자열 컬럼이 {target} — 변경 없음")
            return

        print(f"\n변경 예정: {len(mismatched)}개 컬럼 -> {charset} / {target}")
        if dry_run:
            print("[dry run] 실제 변경 없음. 확인 후 dry_run=False 로 재실행.")
            return
        cur = conn.cursor()
        cur.execute(f"ALTER TABLE {table_name} CONVERT TO CHARACTER SET {charset} COLLATE {target}")
        conn.commit()
        print("ALTER 완료. audit_result_table() 로 [0] 재확인 권장.")
    finally:
        conn.close()


## 3. 실행

In [6]:
# ===== 실행 절차 (위에서 아래로, 1회) =====

db_info = get_db_info()

# ⓪ 진단 — 읽기 전용. 먼저 현재 상태를 눈으로 확인
audit_result_table(db_info)

# ① vintage 키 — 멱등
# migrate_vintage_key(db_info)

# ② 단위 통일 — ⓪의 [3]·[4] 로 방향 확인 후 factor 지정
#    예: 레거시(숫자만 세대)가 '원' 단위, 표준 '천원' -> factor=0.001
# migrate_unit(db_info, factor=0.001, dry_run=True)    # 미리보기
# migrate_unit(db_info, factor=0.001, dry_run=False)   # 실제 반영

# ③ 티커 통일 — 반드시 ①·② 이후
# migrate_ticker(db_info, dry_run=True)                # 충돌 검사 + 미리보기
# migrate_ticker(db_info, dry_run=False)               # 실제 반영

# ④ collation 통일 — ⓪ 의 [0] 에서 불일치가 보이면 실행 (순서 무관 · 멱등)
# migrate_collation(db_info, dry_run=True)             # 현황 + 미리보기
# migrate_collation(db_info, dry_run=False)            # 실제 반영

# 마무리 확인 — 다시 진단해 collation · 키 OK · 형식 단일 · 자릿수 정합을 확인
# audit_result_table(db_info)


AUDIT: korea_revenue_forecast_result

[0] ticker 컬럼 collation (원천 FS 테이블과 일치해야 함)
                   TABLE_NAME COLUMN_NAME CHARACTER_SET_NAME     COLLATION_NAME
        korea_fs_data_from_DG      ticker            utf8mb4 utf8mb4_general_ci
korea_revenue_forecast_result      ticker            utf8mb4 utf8mb4_unicode_ci
    -> 불일치: 아래 [4] 는 명시적 COLLATE 로 우회 중. ④ migrate_collation 으로 통일 권장

[1] UNIQUE KEY
    unique_date_ticker_indicator: (date,ticker,indicator)  <- 구버전 — 실행 시 덮어쓰기 발생

[2] ticker 형식 분포
             fmt  n_rows  n_tickers
A접두사(예: A005930)   51300       1290
  숫자만(예: 005930)   79920       1828
    -> 형식 혼재: 이 상태에서는 단일 형식 조인이 조용히 일부를 놓친다

[3] created_at 세대별 요약 (avg_log10_value 가 세대 간 ~3 차이나면 1000배 의심)
         created_at  fmt  n_rows  avg_log10_value
2025-12-04 22:13:20  숫자만     360            11.49
2025-12-04 22:34:28  숫자만     340            15.93
2025-12-04 22:34:29  숫자만    1100            12.02
2025-12-04 22:45:50  숫자만    1600            11.55
2025-12-04 22:45:51  숫자만  

In [7]:
migrate_vintage_key(db_info)

  [v5] vintage 키 교정: `unique_date_ticker_indicator`(date,ticker,indicator) -> (date,ticker,indicator,created_at)
vintage 키 상태: altered


In [8]:
migrate_unit(db_info, factor=0.001, dry_run=True)

대상: 79,920행  (WHERE ticker NOT LIKE 'A%')  factor=0.001
 id       date ticker indicator        value  value_after
  1 2025-12-31 000020       ETS 1.163060e+11 1.163060e+08
  2 2025-12-31 000020  Ensemble 1.185591e+11 1.185591e+08
  3 2025-12-31 000020    SARIMA 1.171568e+11 1.171568e+08
  4 2025-12-31 000020     Theta 1.222145e+11 1.222145e+08
  5 2026-03-31 000020       ETS 1.192937e+11 1.192937e+08

[dry run] 실제 변경 없음. 확인 후 dry_run=False 로 재실행.


In [9]:
migrate_unit(db_info, factor=0.001, dry_run=False)

대상: 79,920행  (WHERE ticker NOT LIKE 'A%')  factor=0.001
 id       date ticker indicator        value  value_after
  1 2025-12-31 000020       ETS 1.163060e+11 1.163060e+08
  2 2025-12-31 000020  Ensemble 1.185591e+11 1.185591e+08
  3 2025-12-31 000020    SARIMA 1.171568e+11 1.171568e+08
  4 2025-12-31 000020     Theta 1.222145e+11 1.222145e+08
  5 2026-03-31 000020       ETS 1.192937e+11 1.192937e+08

UPDATE 완료: 79,911행


In [10]:
audit_result_table(db_info)

AUDIT: korea_revenue_forecast_result

[0] ticker 컬럼 collation (원천 FS 테이블과 일치해야 함)
                   TABLE_NAME COLUMN_NAME CHARACTER_SET_NAME     COLLATION_NAME
        korea_fs_data_from_DG      ticker            utf8mb4 utf8mb4_general_ci
korea_revenue_forecast_result      ticker            utf8mb4 utf8mb4_unicode_ci
    -> 불일치: 아래 [4] 는 명시적 COLLATE 로 우회 중. ④ migrate_collation 으로 통일 권장

[1] UNIQUE KEY
    unique_date_ticker_indicator_created: (date,ticker,indicator,created_at)  <- OK (vintage)

[2] ticker 형식 분포
             fmt  n_rows  n_tickers
A접두사(예: A005930)   51300       1290
  숫자만(예: 005930)   79920       1828
    -> 형식 혼재: 이 상태에서는 단일 형식 조인이 조용히 일부를 놓친다

[3] created_at 세대별 요약 (avg_log10_value 가 세대 간 ~3 차이나면 1000배 의심)
         created_at  fmt  n_rows  avg_log10_value
2025-12-04 22:13:20  숫자만     360             8.49
2025-12-04 22:34:28  숫자만     340            12.93
2025-12-04 22:34:29  숫자만    1100             9.02
2025-12-04 22:45:50  숫자만    1600             8.55
2025-12-04 22

In [11]:
conn = get_connection(db_info)
diag = pd.read_sql("""
    SELECT created_at, indicator, COUNT(*) AS n,
           ROUND(AVG(LOG10(NULLIF(ABS(value),0))), 2) AS avg_lg,
           ROUND(MIN(LOG10(NULLIF(ABS(value),0))), 2) AS min_lg,
           ROUND(MAX(LOG10(NULLIF(ABS(value),0))), 2) AS max_lg
    FROM korea_revenue_forecast_result
    WHERE ticker NOT LIKE 'A%'
    GROUP BY created_at, indicator
    HAVING avg_lg > 9 OR max_lg > 10
    ORDER BY created_at, indicator
""", conn)
conn.close()
print(diag.to_string(index=False))

         created_at indicator    n  avg_lg  min_lg  max_lg
2025-12-04 22:34:28  Ensemble   85   17.34    7.16   72.33
2025-12-04 22:34:28       ETS   85    8.49    6.95   10.54
2025-12-04 22:34:28    SARIMA   85   17.49    7.58   72.81
2025-12-04 22:34:28     Theta   85    8.41    6.00   10.50
2025-12-04 22:34:29  Ensemble  275    9.78    5.52   66.78
2025-12-04 22:34:29       ETS  275    8.24    5.67   10.70
2025-12-04 22:34:29    SARIMA  275    9.80    5.81   67.25
2025-12-04 22:34:29     Theta  275    8.27    5.48   10.41
2025-12-04 22:45:50  Ensemble  400    9.05    4.95   62.16
2025-12-04 22:45:50       ETS  400    8.05    5.57   10.06
2025-12-04 22:45:50    SARIMA  400    9.08    6.04   62.64
2025-12-04 22:57:09  Ensemble  237    8.24    5.89   39.04
2025-12-04 22:57:09    SARIMA  238    8.26    5.91   39.52
2025-12-04 22:57:10  Ensemble   25   17.90    7.71   44.37
2025-12-04 22:57:10    SARIMA   25   18.02    7.44   44.84
2025-12-04 23:08:45  Ensemble  200   10.47    6.43   76.

In [12]:
RESULT_TICKER_HAS_A_PREFIX = True

In [13]:
migrate_ticker(db_info, dry_run=False)

충돌 없음. 변환 대상 79,920행 -> 표준 'A005930' 형
UPDATE 완료: 79,920행


In [14]:
migrate_collation(db_info, dry_run=False)

앵커: korea_fs_data_from_DG.ticker = utf8mb4 / utf8mb4_general_ci

korea_revenue_forecast_result 문자열 컬럼 현황:
COLUMN_NAME CHARACTER_SET_NAME     COLLATION_NAME
     ticker            utf8mb4 utf8mb4_unicode_ci
  indicator            utf8mb4 utf8mb4_unicode_ci

변경 예정: 2개 컬럼 -> utf8mb4 / utf8mb4_general_ci
ALTER 완료. audit_result_table() 로 [0] 재확인 권장.
